<a href="https://colab.research.google.com/github/21hamza/flyrank_assign_1/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/21hamza/flyrank_assign_1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis + Time Window

**Unit of Analysis**

One row represents the daily search performance of one content page for one client on one specific reporting date.

**Time Window**

For feature exploration and validation, I will use the **March 2026** partition (`month=2026-03`) from the warehouse. This is a mid-panel month that allows feature engineering without using the final month, reducing the risk of leakage from future data.

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""").df()

HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month=2025-02/data_0.parquet' (HTTP 403)

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields Used

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ctr
- content_age_days

These are observable historical signals that are available before the prediction point.

### Label / Proxy
- trend_direction (starter dataset)
- Future decline label (planned for warehouse data)

The starter dataset uses trend direction as a proxy for declining performance.

### Context
- client_hash_id
- content_hash_id
- report_date

These fields are used for grouping, joining, and defining time windows rather than prediction.

### Excluded
- Future-window metrics
- Label-derived columns
- Any information unavailable at prediction time

These are excluded because they would introduce data leakage and produce unrealistic model performance.

In [ ]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 5
""").df()

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Verification Queries

The following queries verify the assumptions made in the data contract.

1. Confirm the grain of the table.
2. Confirm the date range for the selected month.
3. Check data availability using GA4 availability flags.

In [ ]:
# Query 1: Verify grain
con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 5
""").df()

# Query 2: Count rows and date span
con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {TABLES['fact_daily']}
WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""").df()

# Query 3: Availability
con.sql(f"""
SELECT
    COUNT(*) AS ga4_available_rows
FROM {TABLES['fact_daily']}
WHERE ga4_data_available IS TRUE
AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""").df()


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limitations

This dataset has several important limitations.

- Client histories are unbalanced, meaning different clients have different amounts of historical data.
- Some early records contain only Google Search Console data because GA4 tracking started later for certain clients.
- The warehouse is observational and cannot prove that refreshing a page causes future traffic growth.
- Seasonal effects may influence search performance, so a decline does not necessarily indicate a content problem.
- Feature windows and target windows must remain separate to avoid data leakage.

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_rows
FROM {TABLES['fact_daily']}
""").df()

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.